In [1]:
import numpy as np
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from pdf2image import convert_from_path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import json, re, os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

poppler_path = r"C:\Program Files\poppler-24.08.0\Library\bin"

# Step 1: Convert PDF to PIL images
pdf_path = "C:\\Users\\Govind\\Downloads\\Work_And\\Self_Learning\\Invoice_Extraction_demo\\Invoices\\CN-006991.pdf"
file_name = pdf_path.split(sep="\\")[-1].rstrip(".pdf")
pil_images = convert_from_path(pdf_path, dpi = 300, poppler_path = poppler_path)

Output_Images_Path = r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\FirstPageImage"
output_path = os.path.join(Output_Images_Path, f"{file_name}_page_{1}.png")
pil_images[0].save(output_path, 'PNG')

output_dir = os.path.join(os.path.dirname(pdf_path), "OCRed")

# Create OCRed folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Step 2: Convert PIL images to RGB NumPy arrays
image_arrays = [np.array(img.convert("RGB")) for img in pil_images]

# Step 3: Load the OCR model
model = ocr_predictor(pretrained=True)

# Step 4: Run OCR on the image arrays
doc = model(image_arrays)

# Step 5: Visualize + Save OCRed image
for i, (page, pil_img) in enumerate(zip(doc.pages, pil_images)):
    draw = ImageDraw.Draw(pil_img)
    
    for block in page.blocks:
        for line in block.lines:
            # Draw bounding box
            points = (np.array(line.geometry) * [pil_img.width, pil_img.height]).flatten()
            draw.rectangle([points[0], points[1], points[2], points[3]], outline="green", width=2)
            # Optional: draw text near box
            text = " ".join([word.value for word in line.words])
            draw.text((points[0], points[1]-10), text, fill="red")
    
    
    output_path = os.path.join(output_dir, f"{file_name}_page_{i+1}_ocred.png")
    pil_img.save(output_path)
    ## print(f"Saved OCRed image: {output_path}")

# <==========================Required_Fields==============================>
full_text = ""
for page in doc.pages:
    for block in page.blocks:
        for line in block.lines:
            full_text += " ".join([word.value for word in line.words]) + "\n"

def extract_invoice_fields(text: str):
    data = {
        "vendor_name": None,
        "store_name": None,
        "store_id": None,
        "bill_to_store_name": None,
        "bill_to_store_id": None,
        "invoice_number": None,
        "date": None,
        "Total_Items": None,
        "Freight": None,
        "Final_Amount": None,
        "GST": None,
        "type": None
    }

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text_clean = " ".join(lines)

    # Vendor
    for line in lines:
        if not data["vendor_name"] and "PTY" in line.upper():
            data["vendor_name"] = line

    # Type
    if "Credit adjustment note" in full_text_clean:
        data["type"] = "Credit adjustment note"

    # Invoice number
    inv_match = re.search(r"(CN-\d+)", full_text_clean)
    if inv_match:
        data["invoice_number"] = inv_match.group(1)

    # Date
    date_match = re.search(r"(\d{2}/\d{2}/\d{4})", full_text_clean)
    if date_match:
        data["date"] = date_match.group(1)

    # Ship To
    ship_match = re.search(r"Ship To:\s*([^\n]+)", text)
    if ship_match:
        ship_line = ship_match.group(1).strip()
        parts = ship_line.split()
        if len(parts) >= 2:
            data["store_id"] = parts[-1]
            data["store_name"] = " ".join(parts[:-1])

    # Payer
    payer_match = re.search(r"Payer:\s*([^\n]+)", text)
    if payer_match:
        payer_line = payer_match.group(1).strip()
        parts = payer_line.split()
        if len(parts) >= 2:
            data["bill_to_store_id"] = parts[-1]
            data["bill_to_store_name"] = " ".join(parts[:-1])

    # Sub total
    sub_match = re.search(r"TOTAL ITEMS.*?AUD\s*([-+]?\d+\.\d{2})", full_text_clean)
    if sub_match:
        data["Total_Items"] = abs(float(sub_match.group(1)))

    # GST
    gst_match = re.search(r"GST.*?(\d+\.\d{2})\s*%", full_text_clean, re.DOTALL)
    if gst_match:
        data["GST"] = abs(float(gst_match.group(1)))

    # Freight
    freight_match = re.search(r"Freight.*?AUD\s*([-+]?\d+\.\d{2})", full_text_clean)
    if freight_match:
        data["Freight"] = abs(float(freight_match.group(1)))

    # Final total
    final_match = re.search(r"Final Amount.*?AUD\s*([-+]?\d+\.\d{2})", full_text_clean)
    if final_match:
        data["Final_Amount"] = abs(float(final_match.group(1)))

    return data

c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\doctr\models\utils\pytorch.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded v

In [2]:
Extracted_fields = extract_invoice_fields(full_text)

In [3]:
# def validate_numeric_fields(data):
#     issues = []
    
#     # Gather numeric fields (ignore None)
#     numeric_fields = {k: v for k, v in data.items() if isinstance(v, (int, float)) and v is not None}
    
#     # Compare each numeric field with others
#     keys = list(numeric_fields.keys())
#     for i in range(len(keys)):
#         for j in range(i + 1, len(keys)):
#             k1, k2 = keys[i], keys[j]
#             if numeric_fields[k1] == numeric_fields[k2]:
#                 issues.append(f"{k1} matches {k2}, suspicious")
#                 data[k1] = None
#                 data[k2] = None

#     if issues:
#         print("⚠️ Post-extraction validation detected issues:")
#         for issue in issues:
#             print(f" - {issue}")
#         print("Fields reset to None. Re-validation required.")
#         return True
#     else:
#         print("✅ Post-extraction validation passed.")
#         return False

def validate_numeric_fields(data):
    issues = []

    if data is None:
        # By default, assume all numeric-looking fields must be present
        data = []

    # Detect missing required numeric fields
    for field in data:
        if data.get(field) is None:
            issues.append(f"{field} is missing (None)")

    # Gather extracted numeric fields (ignore None)
    numeric_fields = {k: v for k, v in data.items() if isinstance(v, (int, float)) and v is not None}

    # Compare for suspicious equality
    keys = list(numeric_fields.keys())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            k1, k2 = keys[i], keys[j]
            if numeric_fields[k1] == numeric_fields[k2]:
                issues.append(f"{k1} matches {k2}, suspicious")
                data[k1] = None
                data[k2] = None

    if issues:
        print("⚠️ Post-extraction validation detected issues:")
        for issue in issues:
            print(f" - {issue}")
        print("Fields reset to None where applicable. Re-validation required.")
        return True
    else:
        print("✅ Post-extraction validation passed.")
        return False

In [4]:
validation_needed = validate_numeric_fields(Extracted_fields)

⚠️ Post-extraction validation detected issues:
 - Total_Items matches Freight, suspicious
 - Total_Items matches Final_Amount, suspicious
 - Freight matches Final_Amount, suspicious
Fields reset to None where applicable. Re-validation required.


In [5]:
Extracted_fields

{'vendor_name': 'INTEGRIA HEALTHCARE (AUSTRALIA) PTY LIMITED',
 'store_name': 'CWH Glen Waverley',
 'store_id': 'B086',
 'bill_to_store_name': 'CW Management Pty Ltd',
 'bill_to_store_id': 'B655',
 'invoice_number': 'CN-006991',
 'date': '05/10/2022',
 'Total_Items': None,
 'Freight': None,
 'Final_Amount': None,
 'GST': 10.0,
 'type': 'Credit adjustment note'}

In [6]:
validation_needed

True

In [7]:
Extracted_fields.get("vendor_name")

'INTEGRIA HEALTHCARE (AUSTRALIA) PTY LIMITED'

In [8]:
#image_path = "FirstPageImage/B035_CN-016337_20240906_V1127_page_1.png"
poppler_path = r"C:\Program Files\poppler-24.08.0\Library\bin"


In [ ]:
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

# Path to PDF
pdf_path = r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\CN-006991.pdf"

# Step 1: Convert first page of PDF to image
pages = convert_from_path(pdf_path, dpi=300)
page_image = pages[0]  # First page only, index = 0

# Step 2: Crop region (x1, y1, x2, y2)
# Coordinates are in pixels: (left, top, right, bottom)
# Example: top-left (100, 200), width=300, height=100
x1, y1, x2, y2 = 1000, 1500, 10000, 10000
cropped_img = page_image.crop((x1, y1, x2, y2))

# Step 3: OCR only on cropped image
#text = pytesseract.image_to_string(cropped_img, lang='eng')
text = pytesseract.image_to_string(page_image, lang='eng')
print("Extracted text:", text)


Extracted text: Integria”

INTEGRIA HEALTHCARE (AUSTRALIA) PTY LIMITED Customer Care Ph. :

ABN 70096496212

Building 5 Freeway Office Park
2728 Logan Road

Eight Mile Plains QLD 4113

AUS

Ph. 1300 654 336 Fax 1300 654 844

Customer Care Fax :

Email Orders :

1300 654 336
1300 654 844

orders@integria.com

HEALTHCARE ei
Cc Credit adjustment note
. Document No: CN-006991
Ship To: CWH Glen Waverley B086 Payer: CW Management Pty Ltd B655 Date: 05/10/2022
263 Springvale Road Building F, 44 Raglan Street ,
GLEN WAVERLEY VIC 3150 PRESTON VIC 3072 Account No: 95327
AUS AUS Customer Ref:
Delivery No:
Our Order No: SO-0137156
Taken By: Email
. ae F . Discount : “s Dri
Material Brand Description UoM Ordered Shipped RRP List Amount Discount % Unit Price Ext Nett
TMIRONO Thompsons Organic Iron 24mg 30 Tablets EA -1 -1 18.89 9.12 (2.96 32.50% 6.16 -6.16
TMB12TA Thompsons Ultra B12 1000mcg 100 Tablets EA -1 -1 29.19 13.89 (4.51 32.50% 9.38 -9.38
TMTJS30 Thompsons Turmeric Joint Support 30 Tablets 

In [9]:
from doctr.models import ocr_predictor
from doctr.io import DocumentFile

# Load PDF (already returns list of NumPy arrays)
pages = DocumentFile.from_pdf(r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\CN-006991.pdf")

# First page
page_img = pages[0]

# Crop coordinates (y1:y2, x1:x2)
x1, y1, x2, y2 = 800, 700, 1686, 1192
cropped_img = page_img[y1:y2, x1:x2]

# OCR
predictor = ocr_predictor(pretrained=True)
#result = predictor([cropped_img])
result = predictor([cropped_img])
#print(result.export())

doc_dict = result.export()
# ---- Extract text ----
text_parts = []
for page in doc_dict["pages"]:
    for block in page["blocks"]:
        for line in block["lines"]:
            line_text = " ".join(word["value"] for word in line["words"])
            text_parts.append(line_text)

# Join lines into one string
extracted_text = "\n".join(text_parts)
print(extracted_text)

TOTAL ITEMS
AUD
-160.89
Freight
AUD
0.00
GST
10.00 %
-160.89
AUD
-16.09
nd
Final Amount
AUD
-176.98
1/1


In [10]:
page_img.shape

(1192, 1686, 3)

In [11]:
def load_field_map(json_path="FieldLearningMap.json"):
    if os.path.exists(json_path):
        return json.load(open(json_path))
    return {}

def extract_text_by_coords(image, coords):
    x, y, w, h = coords["x"], coords["y"], coords["width"], coords["height"]
    cropped = image.crop((x, y, x + w, y + h))
    return model([np.array(cropped.convert("RGB"))]).pages[0].export()['blocks'][0]['lines']

def Extract_using_coordinates(required_fields, pdf_path):
    vendor = required_fields.get("vendor_name")
    if not vendor:
        print("❌ Vendor name missing. Cannot proceed.")
        return required_fields

    field_map = load_field_map()
    if vendor not in field_map:
        print(f"❌ No template found for vendor: {vendor}")
        return required_fields

    # Convert PDF to image (first page only)
    images = convert_from_path(pdf_path, poppler_path=r"C:\Program Files\poppler-24.08.0\Library\bin")
    image = images[0]

    field_regex = {
        "invoice_number": r"(CN-\d+)",
        "date": r"(\d{2}/\d{2}/\d{4})",
        "Total_Items": r"([-+]?\d+\.\d{2})",
        "Freight": r"([-+]?\d+\.\d{2})",
        "Final_Amount": r"([-+]?\d+\.\d{2})",
        "GST": r"(\d+\.\d{2})\s*%",
    }

    for field, value in required_fields.items():
        if value is not None or field == "vendor_name":
            continue  # Already extracted or not applicable

        found = False
        print(f"🔍 Trying to auto-extract missing field: {field}")
        for template_id, fields in field_map[vendor].items():
            if field not in fields:
                continue

            coords = fields[field]
            try:
                ocr_lines = extract_text_by_coords(image, coords)
                combined_text = " ".join(word['value'] for line in ocr_lines for word in line['words'])
                print(combined_text)

                pattern = field_regex.get(field)
                if pattern:
                    match = re.search(pattern, combined_text)
                    if match:
                        val = match.group(1)
                        required_fields[field] = val if field in ["invoice_number", "date"] else abs(float(val))
                        print(f"✅ Extracted {field}: {required_fields[field]} using template: {template_id}")
                        found = True
                        break
            except Exception as e:
                print(f"⚠️ Failed using template {template_id}: {e}")
                continue

        # if not found:
        #     print(f"⚠️ Manual validation required for: {field}")
        #     os.system("start cmd /k uvicorn Manual_Validation.main:app --reload")
        #     break  # Pause here to allow user to label in the UI

    return required_fields

In [12]:
Extract_using_coordinates(Extracted_fields, pdf_path)

🔍 Trying to auto-extract missing field: Total_Items
🔍 Trying to auto-extract missing field: Freight
🔍 Trying to auto-extract missing field: Final_Amount


{'vendor_name': 'INTEGRIA HEALTHCARE (AUSTRALIA) PTY LIMITED',
 'store_name': 'CWH Glen Waverley',
 'store_id': 'B086',
 'bill_to_store_name': 'CW Management Pty Ltd',
 'bill_to_store_id': 'B655',
 'invoice_number': 'CN-006991',
 'date': '05/10/2022',
 'Total_Items': None,
 'Freight': None,
 'Final_Amount': None,
 'GST': 10.0,
 'type': 'Credit adjustment note'}

In [19]:
import os
import json
import numpy as np
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

# ---------- helpers ----------
def load_field_map(json_path="FieldLearningMap.json"):
    if not os.path.exists(json_path):
        return {}
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def to_xyxy(coords, img_shape):
    """
    Accepts coords in either:
      - {"x":..., "y":..., "width":..., "height":...}  (pixels or 0-1 relative)
      - {"x1":..., "y1":..., "x2":..., "y2":...}       (pixels or 0-1 relative)
    Returns integer (x1, y1, x2, y2) in pixel space, clamped to image bounds.
    """
    H, W = img_shape[:2]

    # Extract as floats
    if all(k in coords for k in ("x", "y", "width", "height")):
        x = float(coords["x"]); y = float(coords["y"])
        w = float(coords["width"]); h = float(coords["height"])
        # If looks like relative (<=1), scale to pixels
        if max(abs(w), abs(h), abs(x), abs(y)) <= 1.0:
            x *= W; y *= H; w *= W; h *= H
        x1, y1, x2, y2 = x, y, x + w, y + h
    elif all(k in coords for k in ("x1", "y1", "x2", "y2")):
        x1 = float(coords["x1"]); y1 = float(coords["y1"])
        x2 = float(coords["x2"]); y2 = float(coords["y2"])
        # If looks like relative (<=1), scale to pixels
        if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.0:
            x1 *= W; x2 *= W; y1 *= H; y2 *= H
    else:
        raise ValueError("Coordinates must have (x,y,width,height) or (x1,y1,x2,y2).")

    # Clamp and cast to int
    x1 = int(max(0, min(W, round(x1))))
    x2 = int(max(0, min(W, round(x2))))
    y1 = int(max(0, min(H, round(y1))))
    y2 = int(max(0, min(H, round(y2))))

    # Ensure proper order
    if x2 < x1: x1, x2 = x2, x1
    if y2 < y1: y1, y2 = y2, y1
    return x1, y1, x2, y2

def ocr_numpy_region(predictor, page_img, coords):
    """
    Crop with NumPy slicing and OCR just that region.
    page_img: np.ndarray (H, W, 3)
    coords: dict from FieldLearningMap.json
    returns plain text
    """
    x1, y1, x2, y2 = to_xyxy(coords, page_img.shape)
    cropped = page_img[y1:y2, x1:x2]          # <-- your preferred y1:y2, x1:x2

    # Run doctr OCR
    doc = predictor([cropped]).export()

    # Flatten to text
    lines = []
    for page in doc["pages"]:
        for block in page["blocks"]:
            for line in block["lines"]:
                lines.append(" ".join(w["value"] for w in line["words"]))
    return " ".join(lines).strip()

# ---------- main ----------
def Extract_using_coordinates(required_fields, pdf_path, page_index=0):
    """
    Returns ONLY OCR text from all fields defined for the vendor
    (no regex / extraction logic here).
    """
    vendor = required_fields.get("vendor_name")
    if not vendor:
        raise ValueError("Vendor name missing in required_fields['vendor_name'].")

    field_map = load_field_map()
    if vendor not in field_map:
        raise ValueError(f"No template found for vendor: {vendor}")

    # Load PDF pages as NumPy arrays (doctr)
    pages = DocumentFile.from_pdf(pdf_path)   # list of np.ndarray (H,W,3)
    if not pages:
        raise ValueError("No pages found in PDF.")
    if page_index >= len(pages):
        raise IndexError(f"PDF has only {len(pages)} pages, got page_index={page_index}.")

    page_img = pages[page_index]

    # Initialize OCR predictor once
    predictor = ocr_predictor(pretrained=True)

    # OCR each field region and return text only
    ocr_texts = {}
    for template_id, fields in field_map[vendor].items():
        for field_name, coords in fields.items():
            # skip vendor_name if present in map
            if field_name == "vendor_name":
                continue
            try:
                text = ocr_numpy_region(predictor, page_img, coords)
                ocr_texts[field_name] = text
            except Exception as e:
                ocr_texts[field_name] = ""  # or keep error if you prefer
                print(f"⚠️ OCR failed for {field_name} ({template_id}): {e}")

    return ocr_texts
Extract_using_coordinates(Extracted_fields, pdf_path, page_index=0)

c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\doctr\models\utils\pytorch.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe

⚠️ OCR failed for Updated_whole (temp_001): division by zero


{'Total_items': 'Freight AUD 0.00 GST 10.00 % -160.89 AUD -16.09 nd Final Amount AUD -176.98 1/1',
 'whole': 'TOTAL ITEMS AUD -160.89 Freight AUD 0.00 GST 10.00 % -160.89 AUD -16.09 nd Final Amount AUD -176.98 1/1',
 'Updated_whole': ''}